In [1]:
import pandas as pd

files = {
    "School Master": "data/cleaned/school_master_cleaned.csv",
    "Attendance": "data/cleaned/student_attendance_cleaned.csv",
    "MDM": "data/cleaned/mid_day_meal_procurement_cleaned.csv",
    "Infrastructure": "data/cleaned/infrastructure_cleaned.csv",
    "Test Scores": "data/cleaned/test_scores_cleaned.csv"
}

for name, path in files.items():
    df = pd.read_csv(path)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print("Shape:", df.shape)
    print("Columns:")
    print(df.columns.tolist())


School Master
Shape: (600, 7)
Columns:
['school_id', 'school_name', 'district', 'block', 'total_enrolled_students', 'school_type', 'medium']

Attendance
Shape: (20000, 12)
Columns:
['record_id', 'date', 'school_id', 'grade', 'total_students', 'present_students', 'teacher_present', 'marked_by', 'attendance_rate', 'impossible_attendance_flag', 'proxy_attendance_flag', 'record_id_missing_flag']

MDM
Shape: (12000, 9)
Columns:
['procurement_id', 'date', 'school_id', 'vendor_name', 'grain_type', 'quantity', 'unit', 'total_cost', 'payment_status']

Infrastructure
Shape: (3000, 10)
Columns:
['inspection_id', 'date', 'school_id', 'has_electricity', 'has_drinking_water', 'has_functional_toilet', 'has_boundary_wall', 'has_playground', 'inspector_name', 'remarks']

Test Scores
Shape: (8000, 9)
Columns:
['assessment_id', 'date', 'school_id', 'grade', 'subject', 'grading_scale', 'score_percentage', 'max_marks', 'total_students_assessed']


In [2]:
import pandas as pd

# Load cleaned datasets
school_master = pd.read_csv("data/cleaned/school_master_cleaned.csv")
attendance = pd.read_csv("data/cleaned/student_attendance_cleaned.csv")
mdm = pd.read_csv("data/cleaned/mid_day_meal_procurement_cleaned.csv")
infrastructure = pd.read_csv("data/cleaned/infrastructure_cleaned.csv")
test_scores = pd.read_csv("data/cleaned/test_scores_cleaned.csv")

# Master school IDs
master_ids = set(school_master["school_id"].dropna().astype(str).str.strip())

datasets = {
    "Attendance": attendance,
    "MDM": mdm,
    "Infrastructure": infrastructure,
    "Test Scores": test_scores
}

print("=" * 70)
print("SCHOOL ID INTEGRATION CHECK")
print("=" * 70)

for name, df in datasets.items():

    ids = set(df["school_id"].dropna().astype(str).str.strip())

    matched = ids.intersection(master_ids)
    unmatched = ids - master_ids

    print(f"\n{name}")
    print("-" * 40)
    print("Unique school IDs:", len(ids))
    print("Matched with School Master:", len(matched))
    print("Unmatched:", len(unmatched))

    if unmatched:
        print("Sample unmatched IDs:", list(unmatched)[:10])

SCHOOL ID INTEGRATION CHECK

Attendance
----------------------------------------
Unique school IDs: 600
Matched with School Master: 600
Unmatched: 0

MDM
----------------------------------------
Unique school IDs: 600
Matched with School Master: 600
Unmatched: 0

Infrastructure
----------------------------------------
Unique school IDs: 598
Matched with School Master: 598
Unmatched: 0

Test Scores
----------------------------------------
Unique school IDs: 600
Matched with School Master: 600
Unmatched: 0


In [3]:
print("=" * 70)
print("DATE INTEGRATION CHECK")
print("=" * 70)

date_columns = {
    "Attendance": "date",
    "MDM": "date",
    "Infrastructure": "date",
    "Test Scores": "date"
}

for name, column in date_columns.items():

    df = datasets[name]

    dates = pd.to_datetime(df[column], errors="coerce")

    print(f"\n{name}")
    print("-" * 40)
    print("Data type:", dates.dtype)
    print("Missing dates:", dates.isna().sum())
    print("Minimum date:", dates.min())
    print("Maximum date:", dates.max())
    print("Invalid dates:", dates.isna().sum())

DATE INTEGRATION CHECK

Attendance
----------------------------------------
Data type: datetime64[ns]
Missing dates: 0
Minimum date: 2025-01-04 00:00:00
Maximum date: 2026-12-03 00:00:00
Invalid dates: 0

MDM
----------------------------------------
Data type: datetime64[ns]
Missing dates: 0
Minimum date: 2025-04-01 00:00:00
Maximum date: 2026-03-31 00:00:00
Invalid dates: 0

Infrastructure
----------------------------------------
Data type: datetime64[ns]
Missing dates: 0
Minimum date: 2025-01-04 00:00:00
Maximum date: 2026-12-03 00:00:00
Invalid dates: 0

Test Scores
----------------------------------------
Data type: datetime64[ns]
Missing dates: 0
Minimum date: 2025-04-01 00:00:00
Maximum date: 2026-03-31 00:00:00
Invalid dates: 0


In [4]:
print("=" * 70)
print("DATASET GRAIN CHECK")
print("=" * 70)

for name, df in {
    "School Master": school_master,
    "Attendance": attendance,
    "MDM": mdm,
    "Infrastructure": infrastructure,
    "Test Scores": test_scores
}.items():

    print(f"\n{name}")
    print("-" * 40)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    if "school_id" in df.columns:
        print("Unique schools:", df["school_id"].nunique())

    if "date" in df.columns:
        print("Unique dates:", df["date"].nunique())

    if "grade" in df.columns:
        print("Unique grades:", df["grade"].nunique())



DATASET GRAIN CHECK

School Master
----------------------------------------
Rows: 600
Columns: 7
Unique schools: 600

Attendance
----------------------------------------
Rows: 20000
Columns: 12
Unique schools: 600
Unique dates: 419
Unique grades: 10

MDM
----------------------------------------
Rows: 12000
Columns: 9
Unique schools: 600
Unique dates: 365

Infrastructure
----------------------------------------
Rows: 3000
Columns: 10
Unique schools: 598
Unique dates: 408

Test Scores
----------------------------------------
Rows: 8000
Columns: 9
Unique schools: 600
Unique dates: 365
Unique grades: 6


In [5]:
print("=" * 70)
print("EXPECTED GRAIN DUPLICATE CHECK")
print("=" * 70)

# Attendance: school + date + grade
attendance_dupes = attendance.duplicated(
    subset=["school_id", "date", "grade"]
).sum()

# MDM: procurement ID should uniquely identify a procurement
mdm_dupes = mdm.duplicated(
    subset=["procurement_id"]
).sum()

# Test Scores: assessment ID should uniquely identify an assessment
test_dupes = test_scores.duplicated(
    subset=["assessment_id"]
).sum()

# Infrastructure: school + date
infra_dupes = infrastructure.duplicated(
    subset=["school_id", "date"]
).sum()

print("Attendance duplicate school-date-grade records:", attendance_dupes)
print("MDM duplicate procurement IDs:", mdm_dupes)
print("Test Scores duplicate assessment IDs:", test_dupes)
print("Infrastructure duplicate school-date records:", infra_dupes)

EXPECTED GRAIN DUPLICATE CHECK
Attendance duplicate school-date-grade records: 94
MDM duplicate procurement IDs: 0
Test Scores duplicate assessment IDs: 0
Infrastructure duplicate school-date records: 14


In [6]:
print("=" * 70)
print("INSPECTING GRAIN DUPLICATES")
print("=" * 70)

# Attendance duplicate groups
attendance_dup_groups = (
    attendance[
        attendance.duplicated(
            subset=["school_id", "date", "grade"],
            keep=False
        )
    ]
    .sort_values(["school_id", "date", "grade"])
)

print("\nATTENDANCE DUPLICATES")
print("-" * 40)
print("Rows involved:", len(attendance_dup_groups))
print("Duplicate groups:",
      attendance_dup_groups[["school_id", "date", "grade"]]
      .drop_duplicates().shape[0])

print(attendance_dup_groups[
    ["school_id", "date", "grade",
     "total_students", "present_students",
     "teacher_present", "marked_by"]
].head(20))


# Infrastructure duplicate groups
infra_dup_groups = (
    infrastructure[
        infrastructure.duplicated(
            subset=["school_id", "date"],
            keep=False
        )
    ]
    .sort_values(["school_id", "date"])
)

print("\n\nINFRASTRUCTURE DUPLICATES")
print("-" * 40)
print("Rows involved:", len(infra_dup_groups))
print("Duplicate groups:",
      infra_dup_groups[["school_id", "date"]]
      .drop_duplicates().shape[0])

print(infra_dup_groups.head(20))

INSPECTING GRAIN DUPLICATES

ATTENDANCE DUPLICATES
----------------------------------------
Rows involved: 188
Duplicate groups: 94
      school_id        date  grade  total_students  present_students  \
2679    SCH0008  2025-12-28      3             171               103   
13926   SCH0008  2025-12-28      3              57                39   
8201    SCH0013  2025-04-27      7             243               235   
10040   SCH0013  2025-04-27      7             241               241   
6084    SCH0021  2025-09-28      2             271               271   
11551   SCH0021  2025-09-28      2             141               141   
6985    SCH0021  2025-12-06      5             257               179   
9012    SCH0021  2025-12-06      5             180               126   
324     SCH0030  2026-03-23      8             123               119   
1558    SCH0030  2026-03-23      8             231               198   
2446    SCH0031  2025-03-06      2             224               140   
8868

In [7]:
print("=" * 70)
print("DUPLICATE GROUP SUMMARY")
print("=" * 70)

# Attendance
attendance_dup_summary = (
    attendance
    .groupby(["school_id", "date", "grade"])
    .size()
    .reset_index(name="record_count")
)

attendance_dup_summary = attendance_dup_summary[
    attendance_dup_summary["record_count"] > 1
]

print("\nATTENDANCE")
print("-" * 40)
print(attendance_dup_summary.to_string(index=False))


# Infrastructure
infra_dup_summary = (
    infrastructure
    .groupby(["school_id", "date"])
    .size()
    .reset_index(name="record_count")
)

infra_dup_summary = infra_dup_summary[
    infra_dup_summary["record_count"] > 1
]

print("\nINFRASTRUCTURE")
print("-" * 40)
print(infra_dup_summary.to_string(index=False))

DUPLICATE GROUP SUMMARY

ATTENDANCE
----------------------------------------
school_id       date  grade  record_count
  SCH0008 2025-12-28      3             2
  SCH0013 2025-04-27      7             2
  SCH0021 2025-09-28      2             2
  SCH0021 2025-12-06      5             2
  SCH0030 2026-03-23      8             2
  SCH0031 2025-03-06      2             2
  SCH0032 2025-10-15      2             2
  SCH0035 2025-11-09      2             2
  SCH0073 2025-12-31      1             2
  SCH0074 2026-03-26      2             2
  SCH0077 2025-08-08      3             2
  SCH0084 2026-02-03      6             2
  SCH0104 2025-08-11      4             2
  SCH0110 2025-08-25      5             2
  SCH0139 2025-09-19      7             2
  SCH0144 2025-09-13      1             2
  SCH0144 2025-11-26      4             2
  SCH0150 2025-08-31      1             2
  SCH0159 2025-09-23      4             2
  SCH0164 2025-06-25      3             2
  SCH0170 2025-02-10      9             2

In [8]:
# Check whether infrastructure grain duplicates are exact duplicates

infra_dups = infrastructure[
    infrastructure.duplicated(
        subset=["school_id", "date"],
        keep=False
    )
].sort_values(["school_id", "date"])

print("Infrastructure duplicate rows:", len(infra_dups))

print("\nExact duplicate rows among these:")
print(
    infra_dups.duplicated(keep=False).sum()
)

print("\nDuplicate groups:")
print(
    infra_dups.groupby(["school_id", "date"]).size()
    .reset_index(name="record_count")
    .to_string(index=False)
)

Infrastructure duplicate rows: 28

Exact duplicate rows among these:
0

Duplicate groups:
school_id       date  record_count
  SCH0033 2026-03-30             2
  SCH0049 2026-01-23             2
  SCH0072 2025-10-22             2
  SCH0199 2025-06-13             2
  SCH0223 2026-03-23             2
  SCH0260 2025-12-15             2
  SCH0265 2025-04-17             2
  SCH0268 2025-12-12             2
  SCH0268 2026-02-20             2
  SCH0321 2025-10-17             2
  SCH0422 2026-03-03             2
  SCH0531 2025-11-12             2
  SCH0551 2025-05-23             2
  SCH0598 2025-06-21             2


In [9]:
# Create the common School Dimension

school_dim = school_master.copy()

# Keep only the columns needed for integration/dashboard
school_dim = school_dim[
    [
        "school_id",
        "school_name",
        "district",
        "block",
        "total_enrolled_students",
        "school_type",
        "medium"
    ]
].copy()

# Validation
print("=" * 70)
print("SCHOOL DIMENSION")
print("=" * 70)

print("Rows:", len(school_dim))
print("Columns:", len(school_dim.columns))
print("Unique school IDs:", school_dim["school_id"].nunique())
print("Duplicate rows:", school_dim.duplicated().sum())
print("Duplicate school IDs:", school_dim["school_id"].duplicated().sum())
print("Missing values:", school_dim.isna().sum().sum())

school_dim.head()

SCHOOL DIMENSION
Rows: 600
Columns: 7
Unique school IDs: 600
Duplicate rows: 0
Duplicate school IDs: 0
Missing values: 0


,school_id,school_name,district,block,total_enrolled_students,school_type,medium
0,SCH0050,Govt. Senior Secondary School Tank,Moga,Moga-I,424,Secondary,Punjabi
1,SCH0583,Govt. Middle School Dara,Ferozepur,Makhu,436,Higher Secondary,Punjabi
2,SCH0083,Govt. Primary School Dutta,Bathinda,Talwandi Sabo,196,Primary,Hindi
3,SCH0306,Govt. Elementary School Bhardwaj,Patiala,Unknown,375,Primary,Hindi
4,SCH0110,Govt. Primary School Goswami,Moga,Bagha Purana,398,Higher Secondary,Punjabi


In [10]:
# Create Monthly Attendance Analytical View

attendance_analysis = attendance.copy()

# Ensure date is datetime
attendance_analysis["date"] = pd.to_datetime(
    attendance_analysis["date"],
    errors="coerce"
)

# Create month
attendance_analysis["month"] = (
    attendance_analysis["date"]
    .dt.to_period("M")
    .astype(str)
)

# Aggregate attendance
attendance_monthly = (
    attendance_analysis
    .groupby(["school_id", "month"], as_index=False)
    .agg(
        total_students_observed=("total_students", "sum"),
        total_students_present=("present_students", "sum"),
        avg_attendance_rate=("attendance_rate", "mean"),
        impossible_attendance_records=("impossible_attendance_flag", "sum"),
        proxy_attendance_records=("proxy_attendance_flag", "sum"),
        attendance_records=("school_id", "size")
    )
)

# Calculate weighted attendance rate
attendance_monthly["attendance_rate"] = (
    attendance_monthly["total_students_present"]
    / attendance_monthly["total_students_observed"]
    * 100
)

# Proxy attendance rate
attendance_monthly["proxy_attendance_rate"] = (
    attendance_monthly["proxy_attendance_records"]
    / attendance_monthly["attendance_records"]
    * 100
)

# Add school information
attendance_monthly = attendance_monthly.merge(
    school_dim,
    on="school_id",
    how="left",
    validate="many_to_one"
)

print("=" * 70)
print("MONTHLY ATTENDANCE ANALYTICAL VIEW")
print("=" * 70)

print("Rows:", len(attendance_monthly))
print("Columns:", len(attendance_monthly.columns))
print("Unique schools:", attendance_monthly["school_id"].nunique())
print("Unique months:", attendance_monthly["month"].nunique())

print("\nMissing values:")
print(attendance_monthly.isna().sum())

print("\nDuplicate school-month records:",
      attendance_monthly.duplicated(
          subset=["school_id", "month"]
      ).sum())

attendance_monthly.head()

MONTHLY ATTENDANCE ANALYTICAL VIEW
Rows: 7645
Columns: 16
Unique schools: 600
Unique months: 24

Missing values:
school_id                         0
month                             0
total_students_observed           0
total_students_present            0
avg_attendance_rate              90
impossible_attendance_records     0
proxy_attendance_records          0
attendance_records                0
attendance_rate                   0
proxy_attendance_rate             0
school_name                       0
district                          0
block                             0
total_enrolled_students           0
school_type                       0
medium                            0
dtype: int64

Duplicate school-month records: 0


,school_id,month,total_students_observed,total_students_present,avg_attendance_rate,impossible_attendance_records,proxy_attendance_records,attendance_records,attendance_rate,proxy_attendance_rate,school_name,district,block,total_enrolled_students,school_type,medium
0,SCH0001,2025-04,248,183,73.790323,0,0,1,73.790323,0.0,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
1,SCH0001,2025-05,308,219,71.458358,0,0,3,71.103896,0.0,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
2,SCH0001,2025-06,265,260,98.407643,0,1,2,98.113208,50.0,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
3,SCH0001,2025-07,411,337,80.934144,0,0,3,81.995134,0.0,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
4,SCH0001,2025-08,689,610,88.365614,0,0,3,88.534107,0.0,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi


In [11]:
# Create Monthly MDM Analytical View

mdm_analysis = mdm.copy()

# Ensure date is datetime
mdm_analysis["date"] = pd.to_datetime(
    mdm_analysis["date"],
    errors="coerce"
)

# Create month
mdm_analysis["month"] = (
    mdm_analysis["date"]
    .dt.to_period("M")
    .astype(str)
)

# Aggregate MDM data
mdm_monthly = (
    mdm_analysis
    .groupby(["school_id", "month"], as_index=False)
    .agg(
        total_mdm_quantity_kg=("quantity", "sum"),
        total_mdm_cost=("total_cost", "sum"),
        procurement_records=("procurement_id", "nunique"),
        unique_vendors=("vendor_name", "nunique")
    )
)

# Add school information
mdm_monthly = mdm_monthly.merge(
    school_dim,
    on="school_id",
    how="left",
    validate="many_to_one"
)

print("=" * 70)
print("MONTHLY MDM ANALYTICAL VIEW")
print("=" * 70)

print("Rows:", len(mdm_monthly))
print("Columns:", len(mdm_monthly.columns))
print("Unique schools:", mdm_monthly["school_id"].nunique())
print("Unique months:", mdm_monthly["month"].nunique())

print("\nMissing values:")
print(mdm_monthly.isna().sum())

print(
    "\nDuplicate school-month records:",
    mdm_monthly.duplicated(
        subset=["school_id", "month"]
    ).sum()
)

mdm_monthly.head()

MONTHLY MDM ANALYTICAL VIEW
Rows: 5847
Columns: 12
Unique schools: 600
Unique months: 12

Missing values:
school_id                  0
month                      0
total_mdm_quantity_kg      0
total_mdm_cost             0
procurement_records        0
unique_vendors             0
school_name                0
district                   0
block                      0
total_enrolled_students    0
school_type                0
medium                     0
dtype: int64

Duplicate school-month records: 0


,school_id,month,total_mdm_quantity_kg,total_mdm_cost,procurement_records,unique_vendors,school_name,district,block,total_enrolled_students,school_type,medium
0,SCH0001,2025-04,78.100,11884.0,5,4,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
1,SCH0001,2025-05,47.700,1908.0,1,1,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
2,SCH0001,2025-06,0.318,2433.0,2,2,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
3,SCH0001,2025-07,0.588,2112.0,2,2,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
4,SCH0001,2025-08,0.276,1242.0,1,1,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi


In [17]:
# ============================================================
# INFRASTRUCTURE PREPARATION
# ============================================================

# Load cleaned infrastructure data
infra_analysis = pd.read_csv(
    "data/cleaned/infrastructure_cleaned.csv"
)

# Convert date to datetime
infra_analysis["date"] = pd.to_datetime(
    infra_analysis["date"],
    errors="coerce"
)

# Create month
infra_analysis["month"] = (
    infra_analysis["date"]
    .dt.to_period("M")
    .astype(str)
)

# Infrastructure indicators
infrastructure_columns = [
    "has_electricity",
    "has_drinking_water",
    "has_functional_toilet",
    "has_boundary_wall",
    "has_playground"
]

# Values considered available
true_values = {
    "true",
    "yes",
    "1",
    "hai",
    "functional",
    "available"
}

# Values considered unavailable
false_values = {
    "false",
    "no",
    "0",
    "nahi",
    "kharab",
    "not available",
    "under repair",
    "broken"
}


def normalize_infrastructure(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if value in true_values:
        return 1

    if value in false_values:
        return 0

    return pd.NA


# Normalize all infrastructure indicators
for column in infrastructure_columns:

    infra_analysis[column] = (
        infra_analysis[column]
        .apply(normalize_infrastructure)
        .astype("Int64")
    )


# Calculate number of amenities checked
infra_analysis["amenities_checked"] = (
    infra_analysis[infrastructure_columns]
    .notna()
    .sum(axis=1)
)

# Calculate number of unavailable amenities
infra_analysis["amenities_missing"] = (
    infra_analysis[infrastructure_columns]
    .eq(0)
    .sum(axis=1)
)

# Calculate infrastructure deficit percentage
infra_analysis["infrastructure_deficit_pct"] = (
    infra_analysis["amenities_missing"]
    / infra_analysis["amenities_checked"]
    * 100
)


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("INFRASTRUCTURE PREPARATION")
print("=" * 70)

print("Rows:", len(infra_analysis))
print("Columns:", len(infra_analysis.columns))

print(
    "Unique schools:",
    infra_analysis["school_id"].nunique()
)

print(
    "Unique months:",
    infra_analysis["month"].nunique()
)

print(
    "Missing dates:",
    infra_analysis["date"].isna().sum()
)

print(
    "Missing deficit values:",
    infra_analysis["infrastructure_deficit_pct"].isna().sum()
)

print(
    "\nInfrastructure deficit range:",
    infra_analysis["infrastructure_deficit_pct"].min(),
    "to",
    infra_analysis["infrastructure_deficit_pct"].max()
)

print("\nInfrastructure indicator values:")

for column in infrastructure_columns:
    print(
        column,
        "→",
        infra_analysis[column].dropna().unique()
    )

INFRASTRUCTURE PREPARATION
Rows: 3000
Columns: 14
Unique schools: 598
Unique months: 24
Missing dates: 0
Missing deficit values: 0

Infrastructure deficit range: 0.0 to 100.0

Infrastructure indicator values:
has_electricity → <IntegerArray>
[1, 0]
Length: 2, dtype: Int64
has_drinking_water → <IntegerArray>
[1, 0]
Length: 2, dtype: Int64
has_functional_toilet → <IntegerArray>
[1, 0]
Length: 2, dtype: Int64
has_boundary_wall → <IntegerArray>
[1, 0]
Length: 2, dtype: Int64
has_playground → <IntegerArray>
[1, 0]
Length: 2, dtype: Int64


In [18]:
# ============================================================
# MONTHLY INFRASTRUCTURE ANALYTICAL VIEW
# ============================================================

infra_monthly = infra_analysis.groupby(
    ["school_id", "month"],
    as_index=False
).agg(
    infrastructure_deficit_pct=(
        "infrastructure_deficit_pct",
        "mean"
    ),

    electricity_availability=(
        "has_electricity",
        "mean"
    ),

    drinking_water_availability=(
        "has_drinking_water",
        "mean"
    ),

    functional_toilet_availability=(
        "has_functional_toilet",
        "mean"
    ),

    boundary_wall_availability=(
        "has_boundary_wall",
        "mean"
    ),

    playground_availability=(
        "has_playground",
        "mean"
    ),

    infrastructure_records=(
        "inspection_id",
        "nunique"
    )
)


# Convert availability ratios to percentages
availability_columns = [
    "electricity_availability",
    "drinking_water_availability",
    "functional_toilet_availability",
    "boundary_wall_availability",
    "playground_availability"
]

for column in availability_columns:
    infra_monthly[column] = (
        infra_monthly[column] * 100
    )


# Add School Master information
infra_monthly = infra_monthly.merge(
    school_dim,
    on="school_id",
    how="left",
    validate="many_to_one"
)


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("MONTHLY INFRASTRUCTURE ANALYTICAL VIEW")
print("=" * 70)

print("Rows:", len(infra_monthly))

print("Columns:", len(infra_monthly.columns))

print(
    "Unique schools:",
    infra_monthly["school_id"].nunique()
)

print(
    "Unique months:",
    infra_monthly["month"].nunique()
)

print(
    "\nDuplicate school-month records:",
    infra_monthly.duplicated(
        subset=["school_id", "month"]
    ).sum()
)

print(
    "\nMissing school information:",
    infra_monthly[
        ["school_name", "district", "block"]
    ].isna().sum().sum()
)

print(
    "\nInfrastructure deficit range:",
    infra_monthly["infrastructure_deficit_pct"].min(),
    "to",
    infra_monthly["infrastructure_deficit_pct"].max()
)

print("\nMissing values:")
print(infra_monthly.isna().sum())

print("\nFirst 5 rows:")
display(infra_monthly.head())

MONTHLY INFRASTRUCTURE ANALYTICAL VIEW
Rows: 2460
Columns: 15
Unique schools: 598
Unique months: 24

Duplicate school-month records: 0

Missing school information: 0

Infrastructure deficit range: 0.0 to 100.0

Missing values:
school_id                           0
month                               0
infrastructure_deficit_pct          0
electricity_availability          103
drinking_water_availability        92
functional_toilet_availability     94
boundary_wall_availability        107
playground_availability            99
infrastructure_records              0
school_name                         0
district                            0
block                               0
total_enrolled_students             0
school_type                         0
medium                              0
dtype: int64

First 5 rows:


,school_id,month,infrastructure_deficit_pct,electricity_availability,drinking_water_availability,functional_toilet_availability,boundary_wall_availability,playground_availability,infrastructure_records,school_name,district,block,total_enrolled_students,school_type,medium
0,SCH0001,2025-07,40.0,100.0,100.0,100.0,0.0,0.0,1,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
1,SCH0001,2025-09,75.0,0.0,100.0,<NA>,0.0,0.0,1,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
2,SCH0002,2025-03,40.0,0.0,100.0,100.0,100.0,0.0,1,Govt. Primary School Dewan,Unknown,Tarn Taran,56,Upper Primary,Hindi
3,SCH0002,2025-09,20.0,100.0,100.0,100.0,100.0,0.0,1,Govt. Primary School Dewan,Unknown,Tarn Taran,56,Upper Primary,Hindi
4,SCH0002,2026-03,0.0,100.0,100.0,100.0,100.0,100.0,1,Govt. Primary School Dewan,Unknown,Tarn Taran,56,Upper Primary,Hindi


In [19]:
# ============================================================
# TEST SCORES ANALYTICAL VIEW
# ============================================================

# Load cleaned test scores
test_analysis = pd.read_csv(
    "data/cleaned/test_scores_cleaned.csv"
)

# Convert date
test_analysis["date"] = pd.to_datetime(
    test_analysis["date"],
    errors="coerce"
)

# Create month
test_analysis["month"] = (
    test_analysis["date"]
    .dt.to_period("M")
    .astype(str)
)

# Ensure grade is numeric
test_analysis["grade"] = pd.to_numeric(
    test_analysis["grade"],
    errors="coerce"
)

# ============================================================
# CREATE SCHOOL + MONTH + GRADE + SUBJECT VIEW
# ============================================================

test_scores_analysis = (
    test_analysis
    .groupby(
        [
            "school_id",
            "month",
            "grade",
            "subject"
        ],
        as_index=False
    )
    .agg(
        average_score_percentage=(
            "score_percentage",
            "mean"
        ),
        students_assessed=(
            "total_students_assessed",
            "sum"
        ),
        assessments=(
            "assessment_id",
            "nunique"
        )
    )
)

# ============================================================
# ADD SCHOOL MASTER INFORMATION
# ============================================================

test_scores_analysis = test_scores_analysis.merge(
    school_dim,
    on="school_id",
    how="left",
    validate="many_to_one"
)

# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("TEST SCORES ANALYTICAL VIEW")
print("=" * 70)

print("Rows:", len(test_scores_analysis))

print(
    "Unique schools:",
    test_scores_analysis["school_id"].nunique()
)

print(
    "Unique months:",
    test_scores_analysis["month"].nunique()
)

print(
    "Unique grades:",
    test_scores_analysis["grade"].nunique()
)

print(
    "Unique subjects:",
    test_scores_analysis["subject"].nunique()
)

print(
    "\nDuplicate school-month-grade-subject records:",
    test_scores_analysis.duplicated(
        subset=[
            "school_id",
            "month",
            "grade",
            "subject"
        ]
    ).sum()
)

print(
    "\nMissing school information:",
    test_scores_analysis[
        ["school_name", "district", "block"]
    ].isna().sum().sum()
)

print(
    "\nScore range:",
    test_scores_analysis[
        "average_score_percentage"
    ].min(),
    "to",
    test_scores_analysis[
        "average_score_percentage"
    ].max()
)

print("\nMissing values:")
print(test_scores_analysis.isna().sum())

print("\nSubjects:")
print(
    sorted(
        test_scores_analysis["subject"]
        .dropna()
        .unique()
    )
)

print("\nFirst 5 rows:")
display(test_scores_analysis.head())

TEST SCORES ANALYTICAL VIEW
Rows: 7845
Unique schools: 600
Unique months: 12
Unique grades: 6
Unique subjects: 6

Duplicate school-month-grade-subject records: 0

Missing school information: 0

Score range: 40.0 to 95.0

Missing values:
school_id                   0
month                       0
grade                       0
subject                     0
average_score_percentage    0
students_assessed           0
assessments                 0
school_name                 0
district                    0
block                       0
total_enrolled_students     0
school_type                 0
medium                      0
dtype: int64

Subjects:
['EVS', 'English', 'Hindi', 'Mathematics', 'Punjabi', 'Science']

First 5 rows:


,school_id,month,grade,subject,average_score_percentage,students_assessed,assessments,school_name,district,block,total_enrolled_students,school_type,medium
0,SCH0001,2025-04,3,Mathematics,53.8,26,1,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
1,SCH0001,2025-04,4,English,55.0,133,1,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
2,SCH0001,2025-04,4,Punjabi,54.0,114,1,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
3,SCH0001,2025-05,3,Punjabi,41.0,124,1,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi
4,SCH0001,2025-05,6,Science,55.0,121,1,Govt. Primary School Nadkarni,Amritsar,Amritsar-I,419,Upper Primary,Hindi


In [20]:
# ============================================================
# SAVE DASHBOARD-READY ANALYTICAL VIEWS
# ============================================================

import os

# Create analytical data folder
os.makedirs("data/analytical", exist_ok=True)

# Save School Dimension
school_dim.to_csv(
    "data/analytical/school_dim.csv",
    index=False
)

# Save Attendance Monthly View
attendance_monthly.to_csv(
    "data/analytical/attendance_monthly.csv",
    index=False
)

# Save MDM Monthly View
mdm_monthly.to_csv(
    "data/analytical/mdm_monthly.csv",
    index=False
)

# Save Infrastructure Monthly View
infra_monthly.to_csv(
    "data/analytical/infrastructure_monthly.csv",
    index=False
)

# Save Test Scores Analytical View
test_scores_analysis.to_csv(
    "data/analytical/test_scores_analysis.csv",
    index=False
)

# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("ANALYTICAL DATASETS SAVED")
print("=" * 70)

files_saved = [
    "data/analytical/school_dim.csv",
    "data/analytical/attendance_monthly.csv",
    "data/analytical/mdm_monthly.csv",
    "data/analytical/infrastructure_monthly.csv",
    "data/analytical/test_scores_analysis.csv"
]

for file in files_saved:
    print("✓", file)

print("\nTotal analytical datasets:", len(files_saved))

ANALYTICAL DATASETS SAVED
✓ data/analytical/school_dim.csv
✓ data/analytical/attendance_monthly.csv
✓ data/analytical/mdm_monthly.csv
✓ data/analytical/infrastructure_monthly.csv
✓ data/analytical/test_scores_analysis.csv

Total analytical datasets: 5


In [21]:
# ============================================================
# FINAL INTEGRATION VALIDATION
# ============================================================

print("=" * 70)
print("FINAL DATA INTEGRATION VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Validate School Dimension
# ------------------------------------------------------------

print("\n1. SCHOOL DIMENSION")
print("-" * 40)

print("Rows:", len(school_dim))
print("Unique schools:", school_dim["school_id"].nunique())
print("Duplicate school IDs:",
      school_dim["school_id"].duplicated().sum())


# ------------------------------------------------------------
# 2. Validate Attendance View
# ------------------------------------------------------------

print("\n2. ATTENDANCE MONTHLY")
print("-" * 40)

print("Rows:", len(attendance_monthly))
print("Unique schools:",
      attendance_monthly["school_id"].nunique())

print(
    "Duplicate school-month:",
    attendance_monthly.duplicated(
        subset=["school_id", "month"]
    ).sum()
)

print(
    "Attendance rate range:",
    round(attendance_monthly["attendance_rate"].min(), 2),
    "to",
    round(attendance_monthly["attendance_rate"].max(), 2)
)


# ------------------------------------------------------------
# 3. Validate MDM View
# ------------------------------------------------------------

print("\n3. MDM MONTHLY")
print("-" * 40)

print("Rows:", len(mdm_monthly))
print("Unique schools:",
      mdm_monthly["school_id"].nunique())

print(
    "Duplicate school-month:",
    mdm_monthly.duplicated(
        subset=["school_id", "month"]
    ).sum()
)

print(
    "Total MDM cost:",
    round(mdm_monthly["total_mdm_cost"].sum(), 2)
)


# ------------------------------------------------------------
# 4. Validate Infrastructure View
# ------------------------------------------------------------

print("\n4. INFRASTRUCTURE MONTHLY")
print("-" * 40)

print("Rows:", len(infra_monthly))
print("Unique schools:",
      infra_monthly["school_id"].nunique())

print(
    "Duplicate school-month:",
    infra_monthly.duplicated(
        subset=["school_id", "month"]
    ).sum()
)

print(
    "Deficit range:",
    round(
        infra_monthly["infrastructure_deficit_pct"].min(),
        2
    ),
    "to",
    round(
        infra_monthly["infrastructure_deficit_pct"].max(),
        2
    )
)


# ------------------------------------------------------------
# 5. Validate Test Scores View
# ------------------------------------------------------------

print("\n5. TEST SCORES")
print("-" * 40)

print("Rows:", len(test_scores_analysis))
print("Unique schools:",
      test_scores_analysis["school_id"].nunique())

print(
    "Duplicate school-month-grade-subject:",
    test_scores_analysis.duplicated(
        subset=[
            "school_id",
            "month",
            "grade",
            "subject"
        ]
    ).sum()
)

print(
    "Score range:",
    round(
        test_scores_analysis[
            "average_score_percentage"
        ].min(),
        2
    ),
    "to",
    round(
        test_scores_analysis[
            "average_score_percentage"
        ].max(),
        2
    )
)


# ------------------------------------------------------------
# 6. Cross-view School ID Validation
# ------------------------------------------------------------

print("\n6. CROSS-VIEW SCHOOL ID VALIDATION")
print("-" * 40)

master_ids = set(school_dim["school_id"])

views = {
    "Attendance": attendance_monthly,
    "MDM": mdm_monthly,
    "Infrastructure": infra_monthly,
    "Test Scores": test_scores_analysis
}

for name, df in views.items():

    ids = set(df["school_id"])

    unmatched = ids - master_ids

    print(
        name,
        "→ Unmatched school IDs:",
        len(unmatched)
    )


# ------------------------------------------------------------
# 7. Overall Result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INTEGRATION STATUS")
print("=" * 70)

print("✓ School dimension validated")
print("✓ Attendance analytical view validated")
print("✓ MDM analytical view validated")
print("✓ Infrastructure analytical view validated")
print("✓ Test Scores analytical view validated")
print("✓ Cross-dataset school relationships validated")
print("\nDATA INTEGRATION COMPLETE")

FINAL DATA INTEGRATION VALIDATION

1. SCHOOL DIMENSION
----------------------------------------
Rows: 600
Unique schools: 600
Duplicate school IDs: 0

2. ATTENDANCE MONTHLY
----------------------------------------
Rows: 7645
Unique schools: 600
Duplicate school-month: 0
Attendance rate range: 58.7 to 144.19

3. MDM MONTHLY
----------------------------------------
Rows: 5847
Unique schools: 600
Duplicate school-month: 0
Total MDM cost: 27915722.0

4. INFRASTRUCTURE MONTHLY
----------------------------------------
Rows: 2460
Unique schools: 598
Duplicate school-month: 0
Deficit range: 0.0 to 100.0

5. TEST SCORES
----------------------------------------
Rows: 7845
Unique schools: 600
Duplicate school-month-grade-subject: 0
Score range: 40.0 to 95.0

6. CROSS-VIEW SCHOOL ID VALIDATION
----------------------------------------
Attendance → Unmatched school IDs: 0
MDM → Unmatched school IDs: 0
Infrastructure → Unmatched school IDs: 0
Test Scores → Unmatched school IDs: 0

INTEGRATION STATUS


In [22]:
# ============================================================
# MONTHLY ATTENDANCE ANALYTICAL VIEW - FINAL VERSION
# ============================================================

attendance_analysis = attendance.copy()

# Convert date
attendance_analysis["date"] = pd.to_datetime(
    attendance_analysis["date"],
    errors="coerce"
)

# Create month
attendance_analysis["month"] = (
    attendance_analysis["date"]
    .dt.to_period("M")
    .astype(str)
)

# ------------------------------------------------------------
# Calculate valid attendance only
# Impossible records are excluded from the attendance-rate
# calculation but retained for anomaly monitoring.
# ------------------------------------------------------------

attendance_analysis["valid_attendance"] = (
    attendance_analysis["impossible_attendance_flag"] == 0
)

attendance_analysis["valid_total_students"] = (
    attendance_analysis["total_students"]
    .where(
        attendance_analysis["valid_attendance"],
        0
    )
)

attendance_analysis["valid_present_students"] = (
    attendance_analysis["present_students"]
    .where(
        attendance_analysis["valid_attendance"],
        0
    )
)

# ------------------------------------------------------------
# Aggregate by school + month
# ------------------------------------------------------------

attendance_monthly = (
    attendance_analysis
    .groupby(
        ["school_id", "month"],
        as_index=False
    )
    .agg(
        total_students_observed=(
            "valid_total_students",
            "sum"
        ),

        total_students_present=(
            "valid_present_students",
            "sum"
        ),

        avg_attendance_rate=(
            "attendance_rate",
            "mean"
        ),

        impossible_attendance_records=(
            "impossible_attendance_flag",
            "sum"
        ),

        proxy_attendance_records=(
            "proxy_attendance_flag",
            "sum"
        ),

        attendance_records=(
            "school_id",
            "size"
        )
    )
)

# ------------------------------------------------------------
# Calculate valid weighted attendance rate
# ------------------------------------------------------------

attendance_monthly["attendance_rate"] = (
    attendance_monthly["total_students_present"]
    / attendance_monthly["total_students_observed"]
    * 100
)

# ------------------------------------------------------------
# Calculate proxy attendance rate
# ------------------------------------------------------------

attendance_monthly["proxy_attendance_rate"] = (
    attendance_monthly["proxy_attendance_records"]
    / attendance_monthly["attendance_records"]
    * 100
)

# ------------------------------------------------------------
# Add School Master information
# ------------------------------------------------------------

attendance_monthly = attendance_monthly.merge(
    school_dim,
    on="school_id",
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 70)
print("FINAL MONTHLY ATTENDANCE ANALYTICAL VIEW")
print("=" * 70)

print("Rows:", len(attendance_monthly))

print(
    "Unique schools:",
    attendance_monthly["school_id"].nunique()
)

print(
    "Unique months:",
    attendance_monthly["month"].nunique()
)

print(
    "Duplicate school-month records:",
    attendance_monthly.duplicated(
        subset=["school_id", "month"]
    ).sum()
)

print(
    "\nAttendance rate range:",
    round(attendance_monthly["attendance_rate"].min(), 2),
    "to",
    round(attendance_monthly["attendance_rate"].max(), 2)
)

print(
    "Attendance rates above 100%:",
    (
        attendance_monthly["attendance_rate"] > 100
    ).sum()
)

print(
    "Attendance rates below 0%:",
    (
        attendance_monthly["attendance_rate"] < 0
    ).sum()
)

print(
    "\nImpossible attendance records retained:",
    attendance_monthly[
        "impossible_attendance_records"
    ].sum()
)

print(
    "Proxy attendance records retained:",
    attendance_monthly[
        "proxy_attendance_records"
    ].sum()
)

print(
    "\nMissing school information:",
    attendance_monthly[
        ["school_name", "district", "block"]
    ].isna().sum().sum()
)

FINAL MONTHLY ATTENDANCE ANALYTICAL VIEW
Rows: 7645
Unique schools: 600
Unique months: 24
Duplicate school-month records: 0

Attendance rate range: 58.7 to 100.0
Attendance rates above 100%: 0
Attendance rates below 0%: 0

Impossible attendance records retained: 806
Proxy attendance records retained: 979

Missing school information: 0


In [23]:
# Save corrected attendance analytical view

attendance_monthly.to_csv(
    "data/analytical/attendance_monthly.csv",
    index=False
)

print("✓ attendance_monthly.csv saved successfully")
print("Rows:", len(attendance_monthly))
print("Columns:", len(attendance_monthly.columns))

✓ attendance_monthly.csv saved successfully
Rows: 7645
Columns: 16


In [28]:
# ============================================================
# FINAL DATA INTEGRATION VALIDATION
# ============================================================

print("=" * 70)
print("FINAL DATA INTEGRATION VALIDATION")
print("=" * 70)

# 1. SCHOOL DIMENSION
print("\n[1] SCHOOL DIMENSION")

print("Rows:", len(school_dim))
print("Unique schools:", school_dim["school_id"].nunique())
print("Duplicate rows:", school_dim.duplicated().sum())
print("Duplicate school IDs:", school_dim["school_id"].duplicated().sum())
print("Missing values:", school_dim.isna().sum().sum())

assert len(school_dim) == 600
assert school_dim["school_id"].nunique() == 600
assert school_dim["school_id"].duplicated().sum() == 0


# 2. ATTENDANCE
print("\n[2] ATTENDANCE MONTHLY")

print("Rows:", len(attendance_monthly))
print("Unique schools:", attendance_monthly["school_id"].nunique())
print("Unique months:", attendance_monthly["month"].nunique())

print(
    "Duplicate school-month:",
    attendance_monthly.duplicated(
        subset=["school_id", "month"]
    ).sum()
)

print(
    "Attendance rate:",
    round(attendance_monthly["attendance_rate"].min(), 2),
    "to",
    round(attendance_monthly["attendance_rate"].max(), 2)
)

print(
    "Rates >100:",
    (attendance_monthly["attendance_rate"] > 100).sum()
)

print(
    "Rates <0:",
    (attendance_monthly["attendance_rate"] < 0).sum()
)

print(
    "Impossible records:",
    attendance_monthly[
        "impossible_attendance_records"
    ].sum()
)

print(
    "Proxy records:",
    attendance_monthly[
        "proxy_attendance_records"
    ].sum()
)

assert attendance_monthly.duplicated(
    subset=["school_id", "month"]
).sum() == 0

assert attendance_monthly["attendance_rate"].max() <= 100
assert attendance_monthly["attendance_rate"].min() >= 0


# 3. MDM
print("\n[3] MDM MONTHLY")

print("Rows:", len(mdm_monthly))
print("Unique schools:", mdm_monthly["school_id"].nunique())
print("Unique months:", mdm_monthly["month"].nunique())

print(
    "Duplicate school-month:",
    mdm_monthly.duplicated(
        subset=["school_id", "month"]
    ).sum()
)

print(
    "Total MDM cost:",
    round(mdm_monthly["total_mdm_cost"].sum(), 2)
)

assert mdm_monthly.duplicated(
    subset=["school_id", "month"]
).sum() == 0


# 4. INFRASTRUCTURE
print("\n[4] INFRASTRUCTURE MONTHLY")

print("Rows:", len(infra_monthly))
print("Unique schools:", infra_monthly["school_id"].nunique())
print("Unique months:", infra_monthly["month"].nunique())

print(
    "Duplicate school-month:",
    infra_monthly.duplicated(
        subset=["school_id", "month"]
    ).sum()
)

print(
    "Infrastructure deficit:",
    round(
        infra_monthly[
            "infrastructure_deficit_pct"
        ].min(),
        2
    ),
    "to",
    round(
        infra_monthly[
            "infrastructure_deficit_pct"
        ].max(),
        2
    )
)

assert infra_monthly.duplicated(
    subset=["school_id", "month"]
).sum() == 0

assert infra_monthly[
    "infrastructure_deficit_pct"
].min() >= 0

assert infra_monthly[
    "infrastructure_deficit_pct"
].max() <= 100


# 5. TEST SCORES
print("\n[5] TEST SCORES")

print("Rows:", len(test_scores_analysis))

print(
    "Unique schools:",
    test_scores_analysis["school_id"].nunique()
)

print(
    "Unique months:",
    test_scores_analysis["month"].nunique()
)

print(
    "Score range:",
    round(
        test_scores_analysis[
            "average_score_percentage"
        ].min(),
        2
    ),
    "to",
    round(
        test_scores_analysis[
            "average_score_percentage"
        ].max(),
        2
    )
)

print(
    "Missing scores:",
    test_scores_analysis[
        "average_score_percentage"
    ].isna().sum()
)

print(
    "Duplicate school-month-grade-subject:",
    test_scores_analysis.duplicated(
        subset=[
            "school_id",
            "month",
            "grade",
            "subject"
        ]
    ).sum()
)

assert test_scores_analysis.duplicated(
    subset=[
        "school_id",
        "month",
        "grade",
        "subject"
    ]
).sum() == 0

assert test_scores_analysis[
    "average_score_percentage"
].min() >= 0

assert test_scores_analysis[
    "average_score_percentage"
].max() <= 100
# 6. CROSS-DATASET SCHOOL IDs
print("\n[6] CROSS-DATASET SCHOOL ID VALIDATION")

master_ids = set(school_dim["school_id"])

datasets = {
    "Attendance": attendance_monthly,
    "MDM": mdm_monthly,
    "Infrastructure": infra_monthly,
    "Test Scores": test_scores_analysis
}

for name, df in datasets.items():

    ids = set(df["school_id"])
    unmatched = ids - master_ids

    print(
        f"{name}: {len(ids)} schools | "
        f"Unmatched: {len(unmatched)}"
    )

    assert len(unmatched) == 0


# FINAL RESULT
print("\n" + "=" * 70)
print("✓ ALL INTEGRATION VALIDATIONS PASSED")
print("=" * 70)

print("\nAnalytical datasets ready for Streamlit:")

print("✓ school_dim.csv")
print("✓ attendance_monthly.csv")
print("✓ mdm_monthly.csv")
print("✓ infrastructure_monthly.csv")
print("✓ test_scores_analysis.csv")

FINAL DATA INTEGRATION VALIDATION

[1] SCHOOL DIMENSION
Rows: 600
Unique schools: 600
Duplicate rows: 0
Duplicate school IDs: 0
Missing values: 0

[2] ATTENDANCE MONTHLY
Rows: 7645
Unique schools: 600
Unique months: 24
Duplicate school-month: 0
Attendance rate: 58.7 to 100.0
Rates >100: 0
Rates <0: 0
Impossible records: 806
Proxy records: 979

[3] MDM MONTHLY
Rows: 5847
Unique schools: 600
Unique months: 12
Duplicate school-month: 0
Total MDM cost: 27915722.0

[4] INFRASTRUCTURE MONTHLY
Rows: 2460
Unique schools: 598
Unique months: 24
Duplicate school-month: 0
Infrastructure deficit: 0.0 to 100.0

[5] TEST SCORES
Rows: 7845
Unique schools: 600
Unique months: 12
Score range: 40.0 to 95.0
Missing scores: 0
Duplicate school-month-grade-subject: 0

[6] CROSS-DATASET SCHOOL ID VALIDATION
Attendance: 600 schools | Unmatched: 0
MDM: 600 schools | Unmatched: 0
Infrastructure: 598 schools | Unmatched: 0
Test Scores: 600 schools | Unmatched: 0

✓ ALL INTEGRATION VALIDATIONS PASSED

Analytical da

In [27]:
print(test_scores_analysis.columns.tolist())

['school_id', 'month', 'grade', 'subject', 'average_score_percentage', 'students_assessed', 'assessments', 'school_name', 'district', 'block', 'total_enrolled_students', 'school_type', 'medium']


In [29]:
print("INTEGRATION STATUS: PASSED")

INTEGRATION STATUS: PASSED
